# f-test ANOVA using statsmodel

In [54]:
import pandas as pd 
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [55]:

four_sessions = pd.DataFrame({
    "Page": ["Page 1"] * 5 + ["Page 2"] * 5 + ["Page 3"] * 5 + ["Page 4"] * 5,
    "Time": [
        164, 172, 177, 156, 195,   # Page 1
        178, 191, 182, 185, 177,   # Page 2
        175, 193, 171, 163, 176,   # Page 3
        155, 166, 164, 170, 168    # Page 4
    ]
})


four_sessions

,Page,Time
0,Page 1,164
1,Page 1,172
2,Page 1,177
3,Page 1,156
4,Page 1,195
5,Page 2,178
6,Page 2,191
7,Page 2,182
8,Page 2,185
9,Page 2,177


In [56]:
g = four_sessions.groupby('Page')['Time'].mean()
g  = g.to_frame()
g.T

Page,Page 1,Page 2,Page 3,Page 4
Time,172.8,182.6,175.6,164.6


In [ ]:
# use OLS
model = smf.ols('Time ~ Page', data=four_sessions).fit()
aov_table = sm.stats.anova_lm(model)
aov_table


,df,sum_sq,mean_sq,F,PR(>F)
Page,3.0,831.4,277.133333,2.739825,0.077586
Residual,16.0,1618.4,101.150000,NaN,NaN


# Manual f-test ANOVA

In [58]:
import numpy as np
import pandas as pd
from scipy.stats import f

In [59]:
four_sessions = pd.DataFrame({
    "Page": ["Page 1"] * 5 + ["Page 2"] * 5 + ["Page 3"] * 5 + ["Page 4"] * 5,
    "Time": [
        164, 172, 177, 156, 195,
        178, 191, 182, 185, 177,
        175, 193, 171, 163, 176,
        155, 166, 164, 170, 168
    ]
})


In [60]:
groups = four_sessions.groupby("Page")["Time"]
group_means = groups.mean()
group_sizes = groups.size()
group_means

Page
Page 1    172.8
Page 2    182.6
Page 3    175.6
Page 4    164.6
Name: Time, dtype: float64

In [61]:
group_sizes

Page
Page 1    5
Page 2    5
Page 3    5
Page 4    5
Name: Time, dtype: int64

In [62]:
overall_mean = four_sessions["Time"].mean()
overall_mean

np.float64(173.9)

In [63]:
k = group_means.shape[0] # number of groups
n = group_sizes.sum() # total number of observations

k, n

(4, np.int64(20))

In [64]:
SSB = sum(
    group_sizes[g] * (group_means[g] - overall_mean) ** 2
    for g in group_means.index
)

In [65]:
SSW = sum(
    ((groups.get_group(g) - group_means[g]) ** 2).sum()
    for g in group_means.index
)

In [66]:
SST = ((four_sessions["Time"] - overall_mean) ** 2).sum()


In [67]:
df_between = k - 1
df_within = n - k

In [ ]:
MS_between = SSB / df_between
MS_within = SSW / df_within

In [69]:

F_stat = MS_between / MS_within

In [ ]:
p_value = 1 - f.cdf(F_stat, df_between, df_within)

In [76]:
results = {
    "SSB": SSB,
    "SSW": SSW,
    "SST": SST,
    "df_between": df_between,
    "df_within": df_within,
    "MS_between": MS_between,
    "MS_within": MS_within,
    "F": F_stat,
    "p_value": p_value
}

df = pd.Series(results, name='f-test Results')
df

SSB            831.400000
SSW           1618.400000
SST           2449.800000
df_between       3.000000
df_within       16.000000
MS_between     277.133333
MS_within      101.150000
F                2.739825
p_value          0.077586
Name: f-test Results, dtype: float64